# Watching the graph grow, and a question travel through it

The real code is in `backend/src/cricketmind_v3/application/graph/`. This notebook only imports it and watches, so it never drifts from the code. Run it again each day: as the stubs become real nodes, every step shows more.

**Kernel:** Select Kernel → Python Environments → the one ending in `backend\.venv\Scripts\python.exe` (or Enter interpreter path: `D:\Cricket\cricketmind-v3-work\cricketmind-v3\backend\.venv\Scripts\python.exe`). The pictures need the VS Code extension *Markdown Preview Mermaid Support*.

In [ ]:
import sys

print(sys.executable)  # should end with backend\.venv\Scripts\python.exe

In [ ]:
from IPython.display import Markdown, display

from cricketmind_v3.application.graph import nodes
from cricketmind_v3.application.graph.build import build_graph
from cricketmind_v3.application.graph.context import GraphContext
from cricketmind_v3.domain.faith.gazetteer import cricket_gazetteer

graph = build_graph()

# what the nodes are given: the names N3 knows (a few, for this notebook)
CONTEXT = GraphContext(
    names=cricket_gazetteer(
        {
            "V Kohli": ("Kohli", "Virat Kohli"),
            "JJ Bumrah": ("Bumrah", "Jasprit Bumrah"),
            "MA Starc": ("Starc", "Mitchell Starc"),
        }
    )
)

# every arrow, read from the real graph; a branch is drawn dotted
arrows = []
for edge in graph.get_graph().edges:
    arrows.append((edge.source, edge.target, edge.conditional))

agents = []
for agent in nodes.AGENTS:
    agents.append(agent.__name__)

# the parts of the v4 figure, in the order a question meets them
PARTS = [
    ["__start__", "n1_video_input", "n2_n5_shot_classifier", "confidence_gate", "n2b_pose_estimator"],
    ["n3_question_reader", "n6_shot_dna"],
    ["n7a_evidence_retrieval", "n7b_context_retrieval", "n8_descriptive_engine", "n9_predictive_engine"],
    ["n10_supervisor_router", *agents],
    ["faith_gate", "n12_report_synthesis", "n13_final_output", "__end__"],
]

# a node the parts forget would never be drawn: say so
placed = []
for part in PARTS:
    placed += part
for name in graph.get_graph().nodes:
    if name not in placed:
        print("not in any part:", name)


def show_part(number):
    """Draw parts 1 to number; the newest part in yellow, a branch dotted."""
    shown = []
    for part in PARTS[:number]:
        shown += part
    lines = ["graph TD"]
    for source, target, branch in arrows:
        if source in shown and target in shown:
            if branch:
                lines.append(f"    {source} -.-> {target}")
            else:
                lines.append(f"    {source} --> {target}")
    for name in PARTS[number - 1]:
        lines.append(f"    class {name} new")
    lines.append("    classDef new fill:#ffd966,stroke:#b8860b,color:#000")
    display(Markdown("```mermaid\n" + "\n".join(lines) + "\n```"))

## Part 1: the clip

The dotted arrows are a branch (Day 56): the video nodes run only when a clip comes with the question. N1 reads the frames. The shot classifier (N2 to N5, your CricShot model) names the shot, the confidence gate checks how sure it is, and the pose estimator (N2b) finds the body's joints, all from the same frames.

In [ ]:
show_part(1)

## Part 2: the question

N3 reads the question, every time: by simple rules since Day 56, by a language model from Day 58. Shot DNA (N6) waits for whoever is still coming: the shot, the body and the question with a clip; the question alone without one.

In [ ]:
show_part(2)

## Part 3: the evidence and the engines

N7a fetches the balls and N7b the context (career, venue). Then the engines compute: N8 counts the past, N9 simulates the future. Both engines wait for both retrievers.

In [ ]:
show_part(3)

## Part 4: the router and the agents

N10 waits for both engines, then sends the question to the agents. For now it sends it to all 13 (12 agents and the Explainer); from Day 73 only to the chosen ones.

In [ ]:
show_part(4)

## Part 5: the checks and the answer: the whole graph

The Faith gate waits for every agent and checks every claim. N12 writes the report from checked claims only, and N13 shows the answer.

In [ ]:
show_part(5)

## The branch: with a clip, and without

The same question twice. Without a clip, the four video nodes never run.

In [ ]:
text = "Kohli out within 6 balls in the T20 powerplay against Bumrah?"

for clip in ("clip.mp4", None):
    final = graph.invoke({"question": text, "clip": clip}, context=CONTEXT)
    skipped = []
    for node in nodes.EVERY_NODE:
        if node.__name__ not in final["visited"]:
            skipped.append(node.__name__)
    print("clip:", clip, "|", len(final["visited"]), "nodes ran | skipped:", skipped)

## One question, step by step

Each line is one node finishing, and what it wrote in the notebook (the state). Look at N3: it writes the subject it read. Nodes that do not wait for each other run side by side.

In [ ]:
question = {"question": text, "clip": None}

for chunk in graph.stream(question, stream_mode="updates", context=CONTEXT):
    for node, update in chunk.items():
        print(node, "->", update)

## Try your own question

Change the text and run it. The rules know only the three players above, the shots, the phases, pace and spin, the formats, "N balls" and "women".

In [ ]:
mine = "Kohli against Bumrah or Starc in ODIs, cover drive or pull?"

final = graph.invoke({"question": mine, "clip": None}, context=CONTEXT)
print(final["subject"])
print(final["answer"])